In [18]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import ast
import numpy as np


In [19]:

# Load data
bt_sentences = pd.read_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/data/bt_sentences_test.csv", index_col=False)
# len(bt_speeches["Speech"])
# bt_speeches20 = bt_speeches[bt_speeches["Period"] == 20]
# bt_speeches20.to_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/data/bundestag_sentences20.csv")
bt_sentences

C:\Users\rober\AppData\Local\Temp\ipykernel_29036\911310729.py:2: DtypeWarning: Columns (0: Name) have mixed types. Specify dtype option on import or set low_memory=False.
  bt_sentences = pd.read_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/data/bt_sentences_test.csv", index_col=False)


,speech_id,sentence_position,sentence,MPID,Date,Chair,Session,Period,Party,Name
0,207,1,Herr Präsident!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
1,207,2,Meine sehr verehrten Damen und Herren!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
2,207,3,Mehr Fortschritt soll es im Lande geben.,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
3,207,4,"Was ist das, und wie soll das gehen?",17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
4,208,1,"Herzlichen Dank, Frau Präsidentin.",17826,2022-01-27,True,14,20,['afd'],Albrecht Glaser
...,...,...,...,...,...,...,...,...,...,...
79047,57040,1,Herr Präsident!,21903,2023-04-19,True,96,20,['linke/pds/sed'],NaN
79048,57040,2,Meine Damen und Herren!,21903,2023-04-19,True,96,20,['linke/pds/sed'],NaN
79049,57040,3,Ein historischer Tag,21903,2023-04-19,True,96,20,['linke/pds/sed'],NaN
79050,57041,1,"Wie begründet die Bundesregierung, dass Entgel...",21904,2023-01-25,False,81,20,['cdu'],NaN


In [20]:
# bt_speeches = bt_speeches.sample(frac=0.001)

# bt_speeches['Speech'] = bt_speeches['Speech'].apply(ast.literal_eval)



In [21]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("luerhard/PopBERT")
model = AutoModelForSequenceClassification.from_pretrained("luerhard/PopBERT")

# CPU inference only
device = torch.device("cpu")
model = model.to(device)
model.eval()


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4469.68it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,

In [22]:

# Optional: let PyTorch use available CPU threads
# Try 4 first, then 6 or 8 and benchmark on your machine
torch.set_num_threads(6)

# Dynamic INT8 quantization for linear layers
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
quantized_model.eval()


C:\Users\rober\AppData\Local\Temp\ipykernel_29036\3820193431.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (key): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (value): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (dropout): Dropout(p=0.1, inplace=False)
  

In [46]:
subset = bt_sentences.iloc[0:50]

sentence_scores = []
for i, sentence in enumerate(subset["sentence"]):
    print(f"analysing sentence {i+1}")
    
    encodings = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.inference_mode():
        out = quantized_model(**encodings)

    probs = torch.sigmoid(out.logits)
    sentence_scores.append(
        [bt_sentences.at[subset.index[i], "speech_id"], 
        bt_sentences.at[subset.index[i], "sentence_position"]] + 
        probs.cpu().numpy()[0].tolist()
    )


analysing sentence 1
analysing sentence 2
analysing sentence 3
analysing sentence 4
analysing sentence 5
analysing sentence 6
analysing sentence 7
analysing sentence 8
analysing sentence 9
analysing sentence 10
analysing sentence 11
analysing sentence 12
analysing sentence 13
analysing sentence 14
analysing sentence 15
analysing sentence 16
analysing sentence 17
analysing sentence 18
analysing sentence 19
analysing sentence 20
analysing sentence 21
analysing sentence 22
analysing sentence 23
analysing sentence 24
analysing sentence 25
analysing sentence 26
analysing sentence 27
analysing sentence 28
analysing sentence 29
analysing sentence 30
analysing sentence 31
analysing sentence 32
analysing sentence 33
analysing sentence 34
analysing sentence 35
analysing sentence 36
analysing sentence 37
analysing sentence 38
analysing sentence 39
analysing sentence 40
analysing sentence 41
analysing sentence 42
analysing sentence 43
analysing sentence 44
analysing sentence 45
analysing sentence 

In [49]:
subset["scores"] = sentence_scores


In [50]:

# Create DataFrame with custom column names
scores_df = pd.DataFrame(sentence_scores, columns=['speech_id', 'sentence_position', 'Anti-Elitism', 'People-Centrism', 'Left-Wing Host-Ideology', 'Right-Wing Host-Ideology'])


In [51]:
scores_df

,speech_id,sentence_position,Anti-Elitism,People-Centrism,Left-Wing Host-Ideology,Right-Wing Host-Ideology
0,207,1,0.133563,0.079781,0.057177,0.071662
1,207,2,0.142408,0.080203,0.056765,0.067925
2,207,3,0.179715,0.136445,0.084267,0.174430
3,207,4,0.121376,0.082099,0.059250,0.081766
4,208,1,0.142205,0.099597,0.075766,0.077304
5,208,2,0.200162,0.114274,0.078428,0.098393
6,209,1,0.153239,0.097394,0.064412,0.088444
7,209,2,0.116895,0.068606,0.054694,0.067936
8,210,1,0.133563,0.079781,0.057177,0.071662
9,210,2,0.149257,0.071118,0.064535,0.069349
